# Gold Layer: Dimension Products (Comment-Driven DQ)

**Pattern**: Metadata-Driven Quality Control via Column Comments  
**Sources**: Silver CRM Product Info

This notebook enforces quality rules defined in the Catalog metadata for your Product data.

In [0]:
%run "../utils/utils_quality_control"

### Step 0: Define Metadata (The 'Invisible' Rules)
Run this after the first execution to 'Tag' your columns. 
Rules: product_id (NOT NULL), list_price (> 0)

In [0]:
target_table = "workspace.gold.dim_products"

if spark.catalog.tableExists(target_table):
    spark.sql(f"ALTER TABLE {target_table} CHANGE COLUMN product_id COMMENT 'DQ: NOT NULL'")
    spark.sql(f"ALTER TABLE {target_table} CHANGE COLUMN list_price COMMENT 'DQ: > 0'")
    print("✓ Product metadata (DQ Tags) updated in Catalog.")
else:
    print("ℹ Table does not exist yet. Run the stream first to create it.")

In [0]:
from pyspark.sql.functions import col, lit
from delta.tables import DeltaTable

# 2. Configuration
quarantine_table = "workspace.gold.quarantine_products"
checkpoint_path = "/Volumes/workspace/bronze/raw_sources/checkpoints/gold_dim_products"

def process_gold_products(batch_df, batch_id):
    table_exists = spark.catalog.tableExists(target_table)
    
    if table_exists:
        # A. Apply Metadata DQ
        validated_df = apply_metadata_dq(batch_df, target_table)
    else:
        print("First Run: Creating target table...")
        validated_df = batch_df.withColumn("is_invalid", lit(False)).withColumn("dq_errors", lit(""))
    
    # B. Split: Clean vs Quarantine
    clean_df = validated_df.filter("is_invalid = false").drop("is_invalid", "dq_errors")
    quarantine_df = validated_df.filter("is_invalid = true")
    
    # C. Save Quarantine
    if quarantine_df.count() > 0:
        print(f"⚠ {quarantine_df.count()} records quarantined.")
        quarantine_df.write.format("delta").mode("append").saveAsTable(quarantine_table)
    
    # D. Save Clean data
    if not table_exists:
        clean_df.write.format("delta").saveAsTable(target_table)
        return

    gold_table = DeltaTable.forName(spark, target_table)
    (gold_table.alias("t")
        .merge(clean_df.alias("s"), "t.product_id = s.product_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())

# 3. Execute Stream
print(f"Starting Comment-Driven Gold processing for {target_table}...")
query = (
    spark.readStream
    .table("workspace.silver.crm_prd_info_cdc")
    .writeStream
    .foreachBatch(process_gold_products)
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()

In [0]:
print(f"✓ Gold processing complete. Total records in {target_table}: {spark.table(target_table).count():,}")

In [0]:
%sql
SELECT *
FROM gold.dim_products
LIMIT 10;